In [8]:
# InitConfig
from time import sleep

from Simulation import (
    HrlAuslagernRequest,
    OpcUaConnectionConfig,
    SkillOA_Simulator,
    SPS_Simulator,
    VsgCompressorControlRequest,
)

START_VSG_AFTER_HRL_SUCCESS = True
ENABLE_SENSOR_LOGGING = True
SENSOR_LOG_POLL_SECONDS = 0.05
SENSOR_LOG_SYMBOLS = [
    "ts_ausleger_vorne",
    "ts_ausleger_hinten",
    "ls_inner",
    "ls_outer",
    "GVL_HRL.HRL_LS_innen",
    "GVL_HRL.HRL_LS_aussen",
    "VSG_SkillSet.VSG_Skill_SuctionProcess.VSG_ErrorDetected",
    "diag_finished",
]

VSG_WORKPIECE_AT_PICKUP = False
SET_VSG_ERROR_FALSE_BEFORE_VSG = True
SET_DIAG_FINISHED_TRUE_BEFORE_VSG = False

opc_config = OpcUaConnectionConfig(
    username="AdminVD",
    password="123456",
)

sps = SPS_Simulator()
skilloa = SkillOA_Simulator(opc_config, sps=sps)

# Notebook-kompatible Hilfsfunktionen.
write_ads_bool = lambda symbol_or_key, value: sps.write_bool(symbol_or_key, value).as_dict()
read_ads_bool = lambda symbol_or_key: sps.read_bool(symbol_or_key).as_dict()
write_ads_string = lambda symbol_or_key, value: sps.write_string(symbol_or_key, value).as_dict()
read_ads_string = lambda symbol_or_key: sps.read_string(symbol_or_key).as_dict()

hrl_request = HrlAuslagernRequest(
    method_call=True,
    horizontal_shelf_i1=1000,
    horizontal_shelf_i2=1000,
    vertical_shelf_i1=500,
    vertical_shelf_i2=500,
    horizontal_transfer_i1=0,
    horizontal_transfer_i2=0,
    vertical_transfer_i1=0,
    vertical_transfer_i2=0,
    timeout_ms=12000,
)

vsg_request = VsgCompressorControlRequest(
    method_call=True,
    destination_reached=True,
    encoder_target_position_01=100,
    encoder_target_position_02=100,
    encoder_target_position_03=200,
    encoder_target_position_04=200,
    encoder_target_position_05=300,
    encoder_target_position_06=300,
)

prepared = sps.prepare_plc_for_skill_tests()
hrl_plan = skilloa.build_hrl_auslagern_start_plan(hrl_request)
vsg_plan = skilloa.build_vsg_compressor_control_start_plan(vsg_request)
write_ads_bool("OPCUA.Diag_finished", False)
write_ads_string("OPCUA.lastFinishedSkill", "")
#sleep(0.5)  # Kurze Pause, damit die SPS den neuen Wert registrieren kann.
#write_ads_bool("VSG_SkillSet.VSG_Skill_SuctionProcess.VSG_ErrorDetected", False)
#write_ads_bool("HRL_SkillSet.ResetButton", False) 

{
    "prepared": prepared,
    "start_vsg_after_hrl_success": START_VSG_AFTER_HRL_SUCCESS,
    "sensor_logging": {
        "enabled": ENABLE_SENSOR_LOGGING,
        "poll_seconds": SENSOR_LOG_POLL_SECONDS,
        "symbols": SENSOR_LOG_SYMBOLS,
    },
    "hrl_plan": {
        "name": hrl_plan.name,
        "method_nodeid": hrl_plan.method.method_nodeid,
        "payload": hrl_plan.payload,
    },
    "vsg_plan": {
        "name": vsg_plan.name,
        "method_nodeid": vsg_plan.method.method_nodeid,
        "payload": vsg_plan.payload,
    },
}


{'prepared': {'sim': True,
  'notaus_a': True,
  'notaus_b': True,
  'hrl_reset_button': False,
  'hrl_start_button': True,
  'hrl_confirmation_button': False,
  'vsg_reset_button': False,
  'vsg_start_button': True,
  'vsg_confirmation_button': False,
  'ts_horizontal': False,
  'ts_vertical': False,
  'ts_ausleger_vorne': False,
  'ts_ausleger_hinten': True,
  'ls_inner': False,
  'ls_outer': False},
 'start_vsg_after_hrl_success': True,
 'sensor_logging': {'enabled': True,
  'poll_seconds': 0.05,
  'symbols': ['ts_ausleger_vorne',
   'ts_ausleger_hinten',
   'ls_inner',
   'ls_outer',
   'GVL_HRL.HRL_LS_innen',
   'GVL_HRL.HRL_LS_aussen',
   'VSG_SkillSet.VSG_Skill_SuctionProcess.VSG_ErrorDetected',
   'diag_finished']},
 'hrl_plan': {'name': 'HRL_Auslagern',
  'method_nodeid': 'ns=4;s=HRL_SkillSet.HRL_NMethod_Auslagern.HRL_NMethod_Auslagern',
  'payload': {'MethodCall': True,
   'HorizontalShelf_I1': 1000,
   'HorizontalShelf_I2': 1000,
   'VerticalShelf_I1': 500,
   'VerticalShelf

In [9]:
# Startbearbeitung
logger_handle = None
hrl_execution = None
hrl_last_finished_write = None
vsg_prepare = None
vsg_manual_writes = []
vsg_execution = None

try:
    if ENABLE_SENSOR_LOGGING:
        logger_handle = sps.start_bool_change_logger(
            SENSOR_LOG_SYMBOLS,
            poll_interval_seconds=SENSOR_LOG_POLL_SECONDS,
            print_prefix="sim_hw",
        )

    print({"step": "start_hrl"})
    hrl_execution = skilloa.start_skill_plan(hrl_plan)
    print(
        {
            "step": "hrl_finished",
            "successful": hrl_execution.is_successful(),
            "outputs": hrl_execution.call.outputs,
        }
    )

    if hrl_execution.is_successful():
        hrl_last_finished_write = write_ads_string(
            "OPCUA.lastFinishedSkill",
            "HRL_NMethod_Auslagern",
        )
        print(
            {
                "step": "write_last_finished_skill",
                "value": "HRL_NMethod_Auslagern",
            }
        )

    if START_VSG_AFTER_HRL_SUCCESS and hrl_execution.is_successful():
        print({"step": "prepare_vsg"})
        vsg_prepare = sps.prepare_vsg_for_compressor_test(
            workpiece_at_pickup=VSG_WORKPIECE_AT_PICKUP
        )

        if SET_VSG_ERROR_FALSE_BEFORE_VSG:
            vsg_manual_writes.append(
                write_ads_bool(
                    "VSG_SkillSet.VSG_Skill_SuctionProcess.VSG_ErrorDetected",
                    False,
                )
            )

        if SET_DIAG_FINISHED_TRUE_BEFORE_VSG:
            vsg_manual_writes.append(write_ads_bool("diag_finished", True))

        print({"step": "start_vsg"})
        vsg_execution = skilloa.start_skill_plan(vsg_plan)
        print(
            {
                "step": "vsg_finished",
                "successful": vsg_execution.is_successful(),
                "outputs": vsg_execution.call.outputs,
            }
        )
    elif START_VSG_AFTER_HRL_SUCCESS:
        print(
            {
                "step": "skip_vsg",
                "reason": "HRL nicht erfolgreich abgeschlossen.",
                "hrl_outputs": None if hrl_execution is None else hrl_execution.call.outputs,
            }
        )
finally:
    if logger_handle is not None:
        stop_event, logger_thread = logger_handle
        stop_event.set()
        logger_thread.join(timeout=1.0)

{
    "hrl": None if hrl_execution is None else hrl_execution.as_dict(),
    "hrl_last_finished_write": hrl_last_finished_write,
    "vsg_prepare": vsg_prepare,
    "vsg_manual_writes": vsg_manual_writes,
    "vsg": None if vsg_execution is None else vsg_execution.as_dict(),
    "final_values": {
        "ls_inner": read_ads_bool("ls_inner"),
        "ls_outer": read_ads_bool("ls_outer"),
        "vsg_error": read_ads_bool("VSG_SkillSet.VSG_Skill_SuctionProcess.VSG_ErrorDetected"),
        "diag_finished": read_ads_bool("diag_finished"),
        "last_finished_skill": read_ads_string("OPCUA.lastFinishedSkill"),
    },
}


{'step': 'start_hrl'}
{'event': 'sim_hw_initial', 'symbol_or_key': 'ts_ausleger_vorne', 'symbol': 'GVL_HRL_Sim.HRL_Ref_Taster_Ausleger_vorne', 'value': False, 'elapsed_seconds': 0.031}
{'event': 'sim_hw_initial', 'symbol_or_key': 'ts_ausleger_hinten', 'symbol': 'GVL_HRL_Sim.HRL_Ref_Taster_Ausleger_hinten', 'value': True, 'elapsed_seconds': 0.031}
{'event': 'sim_hw_initial', 'symbol_or_key': 'ls_inner', 'symbol': 'GVL_HRL_Sim.HRL_LS_innen', 'value': False, 'elapsed_seconds': 0.031}
{'event': 'sim_hw_initial', 'symbol_or_key': 'ls_outer', 'symbol': 'GVL_HRL_Sim.HRL_LS_aussen', 'value': False, 'elapsed_seconds': 0.031}
{'event': 'sim_hw_initial', 'symbol_or_key': 'GVL_HRL.HRL_LS_innen', 'symbol': 'GVL_HRL.HRL_LS_innen', 'value': False, 'elapsed_seconds': 0.031}
{'event': 'sim_hw_initial', 'symbol_or_key': 'GVL_HRL.HRL_LS_aussen', 'symbol': 'GVL_HRL.HRL_LS_aussen', 'value': False, 'elapsed_seconds': 0.031}
{'event': 'sim_hw_initial', 'symbol_or_key': 'VSG_SkillSet.VSG_Skill_SuctionProcess.

{'hrl': {'prepared_signals': {},
  'scheduled_writes': ({'symbol': 'GVL_HRL_Sim.HRL_Ref_Taster_Ausleger_hinten',
    'value': False,
    'delay_seconds': 4.4,
    'scheduled': True},
   {'symbol': 'GVL_HRL_Sim.HRL_Ref_Taster_Ausleger_vorne',
    'value': True,
    'delay_seconds': 5.2,
    'scheduled': True},
   {'symbol': 'GVL_HRL_Sim.HRL_Ref_Taster_Ausleger_vorne',
    'value': False,
    'delay_seconds': 5.5,
    'scheduled': True},
   {'symbol': 'GVL_HRL_Sim.HRL_Ref_Taster_Ausleger_hinten',
    'value': True,
    'delay_seconds': 6.300000000000001,
    'scheduled': True},
   {'symbol': 'GVL_HRL_Sim.HRL_LS_innen',
    'value': True,
    'delay_seconds': 8.7,
    'scheduled': True},
   {'symbol': 'GVL_HRL_Sim.HRL_LS_innen',
    'value': False,
    'delay_seconds': 9.2,
    'scheduled': True},
   {'symbol': 'GVL_HRL_Sim.HRL_LS_aussen',
    'value': True,
    'delay_seconds': 10.0,
    'scheduled': True},
   {'symbol': 'GVL_HRL_Sim.HRL_Enc_horizontal_I1',
    'value': 0,
    'delay_sec

In [7]:
#ErrorReset
write_ads_bool("VSG_SkillSet.ResetButton", True)
write_ads_bool("VSG_SkillSet.VSG_Skill_SuctionProcess.VSG_ErrorDetected", False)
#write_ads_bool("OPCUA.Diag_finished", False)
write_ads_bool("VSG_SkillSet.VSG_DiagnosisHandler.Diag_finished", True)

{
    "vsg_error": read_ads_bool("VSG_SkillSet.VSG_Skill_SuctionProcess.VSG_ErrorDetected"),
    "diag_finished": read_ads_bool("VSG_SkillSet.VSG_DiagnosisHandler.Diag_finished"),
}

{'vsg_error': {'symbol': 'VSG_SkillSet.VSG_Skill_SuctionProcess.VSG_ErrorDetected',
  'value': False},
 'diag_finished': {'symbol': 'VSG_SkillSet.VSG_DiagnosisHandler.Diag_finished',
  'value': False}}